# Summary

Test the Google RAG components


In [1]:
import os, sys

from vertexai import rag
from vertexai.generative_models import GenerativeModel, Tool
import vertexai


utils_path = "../../../../ccc-policy_assistant/interface/utils"
sys.path.insert(0, utils_path)
from authentication import ApiAuthentication

# Set environment variables
dotenv_path = "../../../../ccc-policy_assistant/data/environment"
api_configs = ApiAuthentication(dotenv_path=dotenv_path)
api_configs.set_environ_variables()

# Initialize Vertex AI API once per session
vertexai.init(project=os.environ["GOOGLE_CLOUD_PROJECT"],
              location=os.environ["GOOGLE_CLOUD_LOCATION"],
              staging_bucket=os.environ["STAGING_BUCKET"])




ModuleNotFoundError: No module named 'authentication'

In [2]:
# dotenv_path = "../../../../ccc-policy_assistant/data/environment"
# api_configs = ApiAuthentication(dotenv_path=dotenv_path)
# api_configs.apis_configs

# os.listdir(dotenv_path)
# os.listdir(utils_path)



Found .env file


## Explore existing Corpora (note these are saved in GCP)

In [6]:
# See what methods are available in the rag object
# dir(rag)


# Find existing corpora and delete
existing_corpora = rag.list_corpora()
# print(existing_corpora)

# Get the corpus names and put in a list
existing_corpora_list = []
for existing_corpus in existing_corpora:
    existing_corpora_list.append(existing_corpus.name)
print(existing_corpora_list)

# Delete these
# for corpus_name in existing_corpora_list:
#     deployment.delete_corpus(corpus_name)

# Check that there is nothing there
# for existing_corpus in existing_corpora:
#     existing_corpora_list.append(existing_corpus.name)


['projects/eternal-bongo-435614-b9/locations/us-central1/ragCorpora/1290281293241647104']


## Create and save a corpus

In [5]:
# Create an embedding model
embedding_model_config = rag.RagEmbeddingModelConfig(
    vertex_prediction_endpoint=rag.VertexPredictionEndpoint(
        publisher_model="publishers/google/models/text-embedding-005"
    )
)


In [7]:
# Create a corpus
display_name = "ccc_test_corpus"
rag_corpus = rag.create_corpus(
    display_name=display_name,
    backend_config=rag.RagVectorDbConfig(
        rag_embedding_model_config=embedding_model_config
    ),
)


In [8]:
# Import Files to the RagCorpus
paths = ["gs://ccc-crawl_data"]
rag.import_files(
    rag_corpus.name,
    paths,
    # Optional
    transformation_config=rag.TransformationConfig(
        chunking_config=rag.ChunkingConfig(
            chunk_size=512,
            chunk_overlap=100,
        ),
    ),
    max_embedding_requests_per_min=1000,  # Optional
)


imported_rag_files_count: 1

In [9]:
# Check that the new corpus is there
existing_corpora = rag.list_corpora()
existing_corpora_list = []
for existing_corpus in existing_corpora:
    existing_corpora_list.append(existing_corpus.name)

existing_corpora_list


['projects/eternal-bongo-435614-b9/locations/us-central1/ragCorpora/1290281293241647104']

In [10]:

# Retrieve an existing RAG corpus
rag_corpus = rag.get_corpus(name="projects/eternal-bongo-435614-b9/locations/us-central1/ragCorpora/1290281293241647104")


q1 = "What is the California Community Colleges Chancellor’s Office doing to ensure accessibility of its websites"
q2= "What are LEAP Exams?"


# Direct context retrieval
rag_retrieval_config = rag.RagRetrievalConfig(
    top_k=3,  # Optional
    filter=rag.Filter(vector_distance_threshold=0.5),  # Optional
)
response = rag.retrieval_query(
    rag_resources=[
        rag.RagResource(
            rag_corpus=rag_corpus.name,
            # Optional: supply IDs from `rag.list_files()`.
            # rag_file_ids=["rag-file-1", "rag-file-2", ...],
        )
    ],
    text=q2,
    rag_retrieval_config=rag_retrieval_config,
)
print(response)


contexts {
  contexts {
    source_uri: "gs://ccc-crawl_data/cccco/webpages_pdfs/cccco_webpages_pdfs_wwwccccoedu_2025Feb18_1.txt"
    text: "We share responsibility for creating an equitable, diverse and inclusive community and we see these values as connected to our mission and critical to ensure the well-being of our staff and the students we serve. For more information on the Chancellor\'s Office Diversity, Equity, Inclusion and Accessibility (DEIA) efforts, read our DEIA Strategic Plan (PDF). CalCareers hosts our job postings and assessment examinations for Board of Governors, California Community Colleges. We administer Limited Examination Appointment Program (LEAP) Exams for all our department specific classifications. LEAP exams are an alternate examination and appointment process for the recruitment and hiring of individuals with disabilities into State service. Please contact the Department of Rehabilitation for LEAP certification. Human Resources 916.445.7911 HumanResources@C

In [5]:
dir(response)
# dir(response.contexts)
#
# for r in response.contexts.contexts:
#     print(r.text)
#     print()

import json
# test = json(response.contexts.contexts)

response.contexts
dir(response.contexts.contexts)

res_dict = {}
for i,r in enumerate(response.contexts.contexts):
    res_dict[i] = dict(uri=r.source_uri, text=r.text)


res_dict

res_json = json.dumps(res_dict)
res_json



NameError: name 'response' is not defined

In [21]:
# Enhance generation
# Create a RAG retrieval tool
rag_retrieval_tool = Tool.from_retrieval(
    retrieval=rag.Retrieval(
        source=rag.VertexRagStore(
            rag_resources=[
                rag.RagResource(
                    rag_corpus=rag_corpus.name,  # Currently only 1 corpus is allowed.
                    # Optional: supply IDs from `ccc_agent_workflow.list_files()`.
                    # rag_file_ids=["ccc_agent_workflow-file-1", "ccc_agent_workflow-file-2", ...],
                )
            ],
            rag_retrieval_config=rag_retrieval_config,
        ),
    )
)


In [22]:
# Create a Gemini model instance
rag_model = GenerativeModel(
    model_name="gemini-2.0-flash-001", tools=[rag_retrieval_tool]
)

# Generate response
response = rag_model.generate_content(q1)
print(response.text)


The California Community Colleges Chancellor’s Office (CCCCO) is committed to ensuring digital accessibility for people with disabilities, and is continually improving the user experience for everyone by applying relevant accessibility standards. As of July 2023, the CCCCO websites are partially conformant with WCAG 2.1 level AA, and this has been verified by experts and approved in the Chancellor's Office Certificate of Web Accessibility. The Chancellor’s Office supports present-day technical accessibility standards, such as ARIA 1.2, to ensure interactive web features work correctly with modern browsers and assistive technology and uses an accessibility partner for external evaluation. They have accessibility reports and a Voluntary Product Accessibility Template (VPAT) plan on file for each of their websites, and VPATs are available upon request. The CCCCO welcomes feedback on the accessibility of their websites, and to request an alternate format or report accessibility barriers, y

In [23]:
# Generate response
response = rag_model.generate_content(q2)
print(response.text)



LEAP (Limited Examination Appointment Program) Exams are alternate examination and appointment processes for recruiting and hiring individuals with disabilities into State service. The California Community Colleges Chancellor's Office administers LEAP Exams for their department-specific classifications. For LEAP certification, contact the Department of Rehabilitation.

